In [36]:
!pip install tensorflow==2.13.1 scikit-learn==1.3.0 --upgrade
!pip install pennylane==0.34.0
!pip install nibabel

  Using cached typing_extensions-4.5.0-py3-none-any.whl.metadata (8.5 kB)
Using cached typing_extensions-4.5.0-py3-none-any.whl (27 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pennylane-qiskit 0.43.0 requires qiskit<2.2,>=2.0, which is not installed.
pennylane-qiskit 0.43.0 requires qiskit-aer~=0.17.1, which is not installed.
pennylane-qiskit 0.43.0 requires qiskit-ibm-runtime~=0.41.1, which is not installed.
pennylane-qiskit 0.43.0 requires sympy, which is not installed.
exceptiongroup 1.3.0 requires typing-extensions>=4.6.0; python_version < "3.13", but you have typing-extensions 4.5.0 which is incompatible.
ipython 8.37.0 requires typing_extensions>=4.6; 

In [38]:
import os 
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import numpy as np
import tensorflow.keras.backend as K # Import Keras Backend
import tensorflow.keras.layers as layers # Import Keras Layers for model definition
import pennylane as qml
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from time import time

import tensorflow.keras as tf_keras # Using tf_keras for access to optimizers/models

tf_keras.utils.get_custom_objects().get('Adam').__class__.get_config = lambda self: self.initial_config
tf_keras.utils.get_custom_objects().get('Input').__class__.get_config = lambda self: self.initial_config

N_QUBITS = 4    # Number of qubits for the Quantum Layer
N_CLASSES = 2   # Binary Classification (Tumor vs. No Tumor) for this simulation
IMAGE_SIZE = 28 # Size of simulated 2D image patches
N_SAMPLES = 150 # Increased sample size for better training curve illustration



TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [ ]:
X = np.random.rand(N_SAMPLES, IMAGE_SIZE, IMAGE_SIZE, 1).astype(np.float32)

y_raw = np.random.randint(0, N_CLASSES, N_SAMPLES)
y = to_categorical(y_raw, num_classes=N_CLASSES).astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("--- Simulated Data Shapes ---")
print(f"Training Input Shape: {X_train.shape}")
print(f"Test Input Shape: {X_test.shape}")

--- Simulated Data Shapes ---
Training Input Shape: (105, 28, 28, 1)
Test Input Shape: (45, 28, 28, 1)


In [ ]:
# %%

def build_classical_cnn(input_shape, num_classes):
    """Defines a simple CNN with similar complexity to the hybrid model's classical part."""
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(N_QUBITS, activation='relu'), # Same number of units as N_QUBITS
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Train Classical Model

classical_model = build_classical_cnn(input_shape=(IMAGE_SIZE, IMAGE_SIZE, 1), num_classes=N_CLASSES)

print("--- Classical CNN Baseline Summary ---")
classical_model.summary()

start_time_classical = time()
classical_history = classical_model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=0 # Run silently
)
end_time_classical = time()

loss_c, acc_c = classical_model.evaluate(X_test, y_test, verbose=0)
print(f"\nClassical CNN Test Accuracy: {acc_c*100:.2f}% (Time: {end_time_classical - start_time_classical:.2f}s)")


--- Classical CNN Baseline Summary ---
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_2 (Conv2D)           (None, 28, 28, 16)        160       
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 14, 14, 16)        0         
 g2D)                                                            
                                                                 
 flatten_2 (Flatten)         (None, 3136)              0         
                                                                 
 dense_3 (Dense)             (None, 4)                 12548     
                                                                 
 dense_4 (Dense)             (None, 2)                 10        
                                                                 
Total params: 12718 (49.68 KB)
Trainable params: 12718 (49.68 KB)
Non-trainable p

In [ ]:
@qml.qnode(DEV, interface="tf") # << CHANGED: interface is back to "tf"
def quantum_circuit(inputs, weights):
    """A 4-Qubit VQC for high-dimensional feature transformation."""

    # Feature Encoding (Angle Embedding)
    # Maps 4 classical features to 4 rotation angles (Ry gates)
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation='Y')

    # Variational Ansatz (Learnable structure with entanglement)
    # The weights are optimized during training
    qml.BasicEntanglerLayers(weights, wires=range(N_QUBITS))

    # Measurement (Outputs 4 classical features)
    # Measures the expectation value of the Pauli Z operator on each qubit.
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

weight_shapes = {"weights": (1, N_QUBITS)}

# << CHANGED: Using KerasLayer with interface="tf" is the most tested native approach
# We rely on PennyLane's internal compatibility fix for this combination.
qlayer = qml.qnn.KerasLayer(
    quantum_circuit, 
    weight_shapes, 
    output_dim=N_QUBITS
    # Removed: autograd_home=True, as it is incompatible with this version of PennyLane/TF
)


def build_hybrid_qcnn_model(input_shape, num_classes, qlayer):
    """Classical CNN feature extractor connected to the Quantum Layer classifier. """
    inputs = tf.keras.Input(shape=input_shape)

    # CLASSICAL FEATURE EXTRACTION (Same as the baseline model)
    x = tf.keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = tf.keras.layers.Flatten()(x)

    # DIMENSIONALITY REDUCTION (Feeds exactly 4 features to the 4-qubit circuit)
    quantum_input = tf.keras.layers.Dense(
        N_QUBITS,
        activation='sigmoid',
        name="quantum_input_prep"
    )(x)

    # --- NATIVE KerasLayer INTEGRATION ---
    # PennyLane's KerasLayer handles batching internally when interface="tf" is used.
    quantum_features = qlayer(quantum_input) 

    # FINAL CLASSICAL CLASSIFIER (Output layer)
    output = tf.keras.layers.Dense(num_classes, activation='softmax')(quantum_features)

    model = tf.keras.Model(inputs=inputs, outputs=output)
    return model

hybrid_model = build_hybrid_qcnn_model(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 1),
    num_classes=N_CLASSES,
    qlayer=qlayer
)

hybrid_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- Hybrid QCNN Model Summary ---")
hybrid_model.summary()







--- Hybrid QCNN Model Summary ---
Model: "model_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_11 (InputLayer)       [(None, 28, 28, 1)]       0         
                                                                 
 conv2d_10 (Conv2D)          (None, 28, 28, 16)        160       
                                                                 
 max_pooling2d_10 (MaxPooli  (None, 14, 14, 16)        0         
 ng2D)                                                           
                                                                 
 flatten_10 (Flatten)        (None, 3136)              0         
                                                                 
 quantum_input_prep (Dense)  (None, 4)                 12548     
                                                                 
 keras_layer_2 (KerasLayer)  (None, 4)                 0 (unused)
                        

In [2]:
%matplotlib inline
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import NesterovMomentumOptimizer

dev = qml.device("default.qubit", wires=2)  # Adding wires=2 to match later usage

def variational_classifier(weights, bias, x):
    return circuit(weights, x) + bias

def square_loss(labels, predictions):
    return np.mean((labels - qml.math.stack(predictions)) ** 2)

def accuracy(labels, predictions):
    acc = sum(abs(l - p) < 1e-5 for l, p in zip(labels, predictions))
    acc = acc / len(labels)
    return acc

def get_angles(x):
    beta0 = 2 * np.arcsin(np.sqrt(x[1] ** 2) / np.sqrt(x[0] ** 2 + x[1] ** 2 + 1e-12))
    beta1 = 2 * np.arcsin(np.sqrt(x[3] ** 2) / np.sqrt(x[2] ** 2 + x[3] ** 2 + 1e-12))
    beta2 = 2 * np.arcsin(np.linalg.norm(x[2:]) / np.linalg.norm(x))
    return np.array([beta2, -beta1 / 2, beta1 / 2, -beta0 / 2, beta0 / 2])

def state_preparation(a):
    qml.RY(a[0], wires=0)
    qml.CNOT(wires=[0, 1])
    qml.RY(a[1], wires=1)
    qml.CNOT(wires=[0, 1])
    qml.RY(a[2], wires=1)
    qml.PauliX(wires=0)
    qml.CNOT(wires=[0, 1])
    qml.RY(a[3], wires=1)
    qml.CNOT(wires=[0, 1])
    qml.RY(a[4], wires=1)
    qml.PauliX(wires=0)

x = np.array([0.53896774, 0.79503606, 0.27826503, 0.0], requires_grad=False)
ang = get_angles(x)

@qml.qnode(dev)
def test(angles):
    state_preparation(angles)
    return qml.state()

state = test(ang)
print("x :", np.round(x, 6))
print("angles :", np.round(ang, 6))
print("amplitude vector:", np.round(np.real(state), 6))


x : [0.538968 0.795036 0.278265 0.      ]
angles : [ 0.563975 -0.        0.       -0.975046  0.975046]
amplitude vector: [ 0.538968  0.795036  0.278265 -0.      ]
